# 🦕 DINO SDK v1.2.0 - WorkflowManager Demo

## 🎯 **Gerador de Workflows Databricks para Ingestão**

Este notebook demonstra como usar o **DINO WorkflowManager** para criar jobs/workflows do Databricks que utilizam o **IngestionEngine** com:

✅ **File Arrival Triggers** para automação  
✅ **Job Clusters** com configuração otimizada  
✅ **Custom Tags** para organização  
✅ **Liquid Clustering** e **AutoLoader**  
✅ **Notificações por email**  
✅ **Templates de notebook** gerados automaticamente

---

**📋 Pré-requisitos:**
- Databricks Runtime 13.x+
- Workspace com Unity Catalog
- Permissões para criar jobs
- DINO SDK v1.2.0 instalado

In [ ]:
# Instalar Databricks SDK atualizado e DINO SDK
%pip install --upgrade databricks-sdk==0.49.0
%pip install /Volumes/main/default/system_files/wheels/dino_sdk-1.2.0-py3-none-any.whl --force-reinstall

# Restart Python para garantir que as instalações funcionem
dbutils.library.restartPython()

## 📦 **1. Imports e Configuração Inicial**

In [ ]:
# Imports principais
from dino_sdk import (
    DinoWorkflowManager, 
    DinoWorkflowConfig, 
    create_dino_workflow
)

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import JobSettings

import logging
import json
from pprint import pprint

# Configurar logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("🦕 DINO SDK v1.2.0 - WorkflowManager Demo")
print("=" * 50)
print("✅ Imports realizados com sucesso!")

# Verificar cliente Databricks
try:
    w = WorkspaceClient()
    print(f"✅ Workspace Client conectado: {w.config.host}")
except Exception as e:
    print(f"❌ Erro na conexão: {e}")

## 🏗️ **2. Configuração de Cluster para Jobs**

Vamos definir a configuração padrão de cluster que será usada nos jobs DINO.

In [ ]:
# Configuração de exemplo de cluster
cluster_config_example = {
    "data_security_mode": "DATA_SECURITY_MODE_DEDICATED",
    "custom_tags": {
        "Projeto": "DINO_SDK",
        "Catalogo": "main",
        "Schema": "bronze", 
        "Tabela": "vendas"
    },
    "spark_env_vars": {
        "PYSPARK_PYTHON": "/databricks/python3/bin/python3"
    },
    "azure_attributes": {
        "availability": "SPOT_WITH_FALLBACK_AZURE"
    },
    "runtime_engine": "PHOTON",
    "spark_version": "17.1.x-scala2.13",
    "node_type_id": "Standard_D4ds_v5",
    "is_single_node": False,
    "autoscale": {
        "min_workers": 1,
        "max_workers": 2
    }
}

print("🔧 Configuração de Cluster DINO:")
pprint(cluster_config_example)

## 🚀 **3. Criando WorkflowManager - Método Simples**

Vamos começar com o método mais simples usando a função helper `create_dino_workflow`.

In [ ]:
# Exemplo 1: Job MANUAL (scheduled)
print("🔧 Exemplo 1: Workflow Manual com Schedule")
print("=" * 45)

manual_result = create_dino_workflow(
    job_name="dino-ingest-vendas-bronze-manual",
    notebook_path="/Workspace/Users/user@company.com/dino_ingestion_notebook",
    catalog_name="main",
    schema_name="bronze",
    table_name="vendas",
    source_path="abfss://container@storage.dfs.core.windows.net/raw/vendas/",
    
    # Configurações de automação
    is_automated=False,  # Job manual/scheduled
    cron_schedule="0 0 8 * * ?",  # 8h da manhã todos os dias
    
    # Configurações de ingestão
    liquid_clustering=True,
    clustering_columns=["region", "date"],
    schema_evolution_mode="rescue",
    
    # Configurações de cluster
    node_type_id="Standard_D4ds_v5",
    min_workers=1,
    max_workers=3,
    
    # Configurações extras
    projeto="Vendas_Analytics"
)

print("📋 Resultado do Job Manual:")
pprint(manual_result)

In [ ]:
# Exemplo 2: Job AUTOMATIZADO com File Arrival Trigger
print("\n🔧 Exemplo 2: Workflow Automatizado com File Arrival")
print("=" * 50)

# IMPORTANTE: Para jobs automatizados, is_automated=True e file_arrival_url é obrigatório
automated_result = create_dino_workflow(
    job_name="dino-ingest-sales-bronze-auto",
    notebook_path="/Workspace/Users/user@company.com/dino_auto_notebook",
    catalog_name="main",
    schema_name="bronze", 
    table_name="sales_data",
    source_path="abfss://storage@account.dfs.core.windows.net/raw/sales/",
    
    # 🔥 AUTOMAÇÃO ATIVA - File Arrival Trigger
    is_automated=True,
    file_arrival_url="abfss://storage@account.dfs.core.windows.net/raw/sales/",
    
    # Configurações avançadas
    liquid_clustering=True,
    clustering_columns=["customer_id", "order_date", "region"],
    schema_evolution_mode="addNewColumns",
    type_run="streaming",  # Streaming para jobs automatizados
    
    # Notificações
    email_notifications={
        "on_failure": ["admin@company.com", "team@company.com"],
        "on_success": ["team@company.com"]
    }
)

print("📋 Resultado do Job Automatizado:")
pprint(automated_result)

## ⚙️ **4. Método Avançado - DinoWorkflowConfig**

Para configurações mais complexas, use a classe `DinoWorkflowConfig` diretamente.

In [ ]:
# Configuração avançada usando DinoWorkflowConfig
print("🔧 Exemplo 3: Configuração Avançada")
print("=" * 40)

# Criar configuração detalhada
advanced_config = DinoWorkflowConfig(
    # Informações básicas
    job_name="dino-advanced-pipeline-customer-data",
    notebook_path="/Workspace/Users/user@company.com/advanced_pipeline_notebook",
    
    # Parâmetros de ingestão
    catalog_name="production", 
    schema_name="bronze",
    table_name="customer_events",
    source_path="abfss://prod-data@company.dfs.core.windows.net/events/customers/",
    
    # Automação com file arrival
    is_automated=True,
    file_arrival_url="abfss://prod-data@company.dfs.core.windows.net/events/customers/",
    
    # Configurações de cluster personalizadas
    node_type_id="Standard_D8ds_v5",  # Cluster maior
    min_workers=2,
    max_workers=8,  # Mais escalabilidade
    spark_version="17.1.x-scala2.13",
    is_single_node=False,
    
    # Configurações de ingestão avançadas
    liquid_clustering=True,
    clustering_columns=["customer_segment", "event_date", "region"],
    schema_evolution_mode="rescue",
    type_run="streaming",
    
    # Notificações completas
    email_notifications={
        "on_start": ["ops@company.com"],
        "on_success": ["data-team@company.com", "analytics@company.com"], 
        "on_failure": ["alerts@company.com", "ops@company.com", "data-team@company.com"]
    },
    
    # Configurações extras
    timezone="America/Sao_Paulo",
    projeto="Customer_Analytics_Pipeline"
)

print("📋 Configuração Avançada Criada:")
print(f"   • Job Name: {advanced_config.job_name}")
print(f"   • Destino: {advanced_config.catalog_name}.{advanced_config.schema_name}.{advanced_config.table_name}")
print(f"   • Automação: {advanced_config.is_automated}")
print(f"   • Clustering: {advanced_config.clustering_columns}")
print(f"   • Tipo Run: {advanced_config.type_run}")

In [ ]:
# Usar o WorkflowManager diretamente para criar o job avançado
print("🚀 Criando Job Avançado...")

manager = DinoWorkflowManager()
advanced_result = manager.create_workflow(advanced_config)

print("📋 Resultado do Job Avançado:")
pprint(advanced_result)

if advanced_result['success']:
    print(f"\n🎉 Job criado com sucesso!")
    print(f"🔗 URL: {advanced_result['job_url']}")
    print(f"🆔 Job ID: {advanced_result['job_id']}")
else:
    print(f"\n❌ Erro na criação: {advanced_result['error']}")

## 📝 **5. Gerando Template de Notebook**

O WorkflowManager pode gerar automaticamente templates de notebook otimizados para cada configuração.

In [ ]:
# Gerar template de notebook para o job avançado
print("📝 Gerando Template de Notebook...")
print("=" * 35)

notebook_template = manager.create_notebook_template(advanced_config)

# Mostrar as primeiras linhas do template
template_lines = notebook_template.split('\n')
print("📋 Primeiras 30 linhas do template gerado:")
print("=" * 45)

for i, line in enumerate(template_lines[:30], 1):
    print(f"{i:2d}: {line}")

print("\n...")
print(f"📊 Template completo tem {len(template_lines)} linhas")

In [ ]:
# Salvar template em arquivo (opcional)
template_path = "/tmp/dino_notebook_template.py"

with open(template_path, 'w', encoding='utf-8') as f:
    f.write(notebook_template)

print(f"💾 Template salvo em: {template_path}")
print("📋 Você pode copiar este código para criar seu notebook no Databricks")

## 📊 **6. Gerenciamento de Jobs DINO**

Vamos ver como listar e monitorar jobs DINO existentes.

In [ ]:
# Listar todos os jobs DINO no workspace
print("📊 Listando Jobs DINO no Workspace...")
print("=" * 40)

dino_jobs = manager.list_dino_jobs()

if dino_jobs:
    print(f"🦕 Encontrados {len(dino_jobs)} jobs DINO:")
    for job in dino_jobs:
        print(f"   • {job['job_name']} (ID: {job['job_id']})")
        print(f"     Criador: {job['creator']}")
        print(f"     Criado em: {job['created_time']}")
        print()
else:
    print("ℹ️ Nenhum job DINO encontrado no workspace")

In [ ]:
# Verificar status de jobs específicos (se existirem)
if dino_jobs:
    print("🔍 Verificando Status dos Jobs DINO...")
    print("=" * 40)
    
    for job in dino_jobs[:3]:  # Verificar até 3 jobs
        job_status = manager.get_job_status(job['job_id'])
        
        print(f"📋 Status do Job: {job['job_name']}")
        print(f"   • ID: {job_status['job_id']}")
        print(f"   • Status: {job_status.get('status', 'Unknown')}")
        print(f"   • Execuções recentes: {job_status.get('recent_runs', 0)}")
        print(f"   • Último estado: {job_status.get('last_run_state', 'Never run')}")
        print()

## 🧪 **7. Validação e Troubleshooting**

In [ ]:
# Teste de validação de configuração
print("🧪 Testando Validação de Configuração...")
print("=" * 45)

# Exemplo de configuração INVÁLIDA (para testar validação)
try:
    invalid_config = DinoWorkflowConfig(
        job_name="",  # Nome vazio - inválido
        notebook_path="",  # Path vazio - inválido
        catalog_name="main",
        schema_name="bronze",
        table_name="test",
        source_path="abfss://test/",
        is_automated=True,
        # file_arrival_url não fornecido - inválido para jobs automatizados
    )
    
    result = manager.create_workflow(invalid_config)
    print("📋 Resultado da configuração inválida:")
    pprint(result)
    
except Exception as e:
    print(f"✅ Validação funcionando - erro capturado: {e}")

In [ ]:
# Exemplo de configuração VÁLIDA com settings mínimos
print("\n🧪 Testando Configuração Mínima Válida...")
print("=" * 45)

minimal_config = DinoWorkflowConfig(
    job_name="dino-test-minimal",
    notebook_path="/Workspace/Users/test@company.com/minimal_notebook",
    catalog_name="main",
    schema_name="bronze",
    table_name="test_table",
    source_path="abfss://test@storage.dfs.core.windows.net/data/",
    # Usar configurações padrão para tudo mais
)

print("📋 Configuração Mínima:")
print(f"   • Job: {minimal_config.job_name}")
print(f"   • Automação: {minimal_config.is_automated}")
print(f"   • Schedule: {minimal_config.cron_schedule}")
print(f"   • Cluster: {minimal_config.node_type_id}")
print(f"   • Workers: {minimal_config.min_workers}-{minimal_config.max_workers}")

## 🎯 **8. Casos de Uso Reais**

Vamos criar alguns exemplos de workflows para casos de uso comuns.

In [ ]:
# Caso de Uso 1: Ingestão de logs em tempo real
print("🎯 Caso de Uso 1: Ingestão de Logs em Tempo Real")
print("=" * 50)

logs_workflow = create_dino_workflow(
    job_name="dino-ingest-application-logs",
    notebook_path="/Workspace/Shared/DINO/logs_ingestion_notebook", 
    catalog_name="observability",
    schema_name="raw_logs",
    table_name="application_events",
    source_path="abfss://logs@company.dfs.core.windows.net/applications/",
    
    # Automação para logs contínuos
    is_automated=True,
    file_arrival_url="abfss://logs@company.dfs.core.windows.net/applications/",
    
    # Configurações otimizadas para logs
    liquid_clustering=True,
    clustering_columns=["application", "log_level", "timestamp_hour"],
    schema_evolution_mode="rescue",  # Capturar novos campos de log
    type_run="streaming",
    
    # Cluster menor para logs
    node_type_id="Standard_D4ds_v5",
    min_workers=1,
    max_workers=4
)

print(f"📋 Logs Workflow: {logs_workflow.get('success', False)}")

In [ ]:
# Caso de Uso 2: ETL de dados financeiros (batch diário)
print("🎯 Caso de Uso 2: ETL Financeiro Diário")
print("=" * 40)

financial_workflow = create_dino_workflow(
    job_name="dino-ingest-financial-transactions",
    notebook_path="/Workspace/Finance/daily_transactions_notebook",
    catalog_name="finance",
    schema_name="bronze",
    table_name="daily_transactions",
    source_path="abfss://finance@company.dfs.core.windows.net/transactions/daily/",
    
    # Schedule diário às 6h da manhã
    is_automated=False,
    cron_schedule="0 0 6 * * ?",  # 6h da manhã
    
    # Configurações para dados financeiros
    liquid_clustering=True,
    clustering_columns=["transaction_date", "account_type", "currency"],
    schema_evolution_mode="strict",  # Não permitir mudanças de schema
    type_run="batch",
    
    # Cluster maior para processamento financeiro
    node_type_id="Standard_D8ds_v5",
    min_workers=2,
    max_workers=6,
    
    # Notificações críticas
    email_notifications={
        "on_failure": ["finance-alerts@company.com", "ops@company.com"],
        "on_success": ["finance-team@company.com"]
    }
)

print(f"📋 Financial Workflow: {financial_workflow.get('success', False)}")

In [ ]:
# Caso de Uso 3: Ingestão de dados de IoT (streaming contínuo)
print("🎯 Caso de Uso 3: Dados IoT Streaming")
print("=" * 35)

iot_workflow = create_dino_workflow(
    job_name="dino-ingest-iot-sensors",
    notebook_path="/Workspace/IoT/sensors_streaming_notebook",
    catalog_name="iot_platform", 
    schema_name="sensor_data",
    table_name="device_telemetry",
    source_path="abfss://iot@sensors.dfs.core.windows.net/telemetry/",
    
    # Automação para dados IoT contínuos
    is_automated=True,
    file_arrival_url="abfss://iot@sensors.dfs.core.windows.net/telemetry/",
    
    # Otimizações para IoT
    liquid_clustering=True,
    clustering_columns=["device_type", "location", "timestamp_hour"],
    schema_evolution_mode="addNewColumns",  # Novos sensores podem adicionar campos
    type_run="streaming",
    
    # Cluster otimizado para streaming
    node_type_id="Standard_D4ds_v5",
    min_workers=2,
    max_workers=10,  # Escala para picos de dados
    
    # Projeto específico
    projeto="IoT_Platform_Ingestion"
)

print(f"📋 IoT Workflow: {iot_workflow.get('success', False)}")

## 🎉 **9. Resumo e Próximos Passos**

In [ ]:
# Resumo final da demonstração
print("🎉 DINO SDK v1.2.0 - WorkflowManager Demo Completa!")
print("=" * 55)

demo_summary = {
    "workflows_criados": 0,
    "workflows_sucesso": 0,
    "workflows_erro": 0,
    "tipos_testados": []
}

# Contar resultados (simulado)
results = [manual_result, automated_result, advanced_result, logs_workflow, financial_workflow, iot_workflow]

for result in results:
    demo_summary["workflows_criados"] += 1
    if result.get('success', False):
        demo_summary["workflows_sucesso"] += 1
    else:
        demo_summary["workflows_erro"] += 1

demo_summary["tipos_testados"] = [
    "Manual Scheduled",
    "File Arrival Automated", 
    "Advanced Configuration",
    "Real-time Logs",
    "Financial ETL",
    "IoT Streaming"
]

print("📊 RESUMO DA DEMONSTRAÇÃO:")
print(f"   ✅ Workflows criados: {demo_summary['workflows_criados']}")
print(f"   ✅ Sucessos: {demo_summary['workflows_sucesso']}")  
print(f"   ❌ Erros: {demo_summary['workflows_erro']}")

print(f"\n🎯 TIPOS DE WORKFLOW TESTADOS:")
for tipo in demo_summary["tipos_testados"]:
    print(f"   • {tipo}")

print(f"\n🚀 FUNCIONALIDADES DEMONSTRADAS:")
features = [
    "File Arrival Triggers automáticos",
    "Job Clusters com configuração otimizada", 
    "Custom Tags para organização",
    "Liquid Clustering e AutoLoader",
    "Templates de notebook automáticos",
    "Notificações por email",
    "Validação de configuração",
    "Gerenciamento de jobs existentes"
]

for feature in features:
    print(f"   ✅ {feature}")

## 📋 **Próximos Passos**

### **Para usar o WorkflowManager em produção:**

1. **📝 Criar Notebooks**: Use os templates gerados para criar seus notebooks de ingestão
2. **🔧 Configurar Paths**: Ajuste os caminhos dos notebooks e dados para seu ambiente
3. **📧 Setup Notificações**: Configure emails para monitoramento
4. **🔐 Configurar Secrets**: Setup Azure credentials e conexões
5. **🧪 Testar Workflows**: Execute jobs em ambiente de desenvolvimento primeiro
6. **📊 Monitorar**: Use `list_dino_jobs()` e `get_job_status()` para acompanhar

### **Documentação Completa:**
- **README.md**: Guia completo do DINO SDK
- **examples/**: Exemplos práticos de uso
- **notebooks/**: Notebooks de demonstração

### **🦕 DINO SDK v1.2.0 - Simplifique seus workflows no Databricks!**